### ScrapeMonthlyProductionSIAP.py -- Jay Sayre, jsayre@ucdavis.edu

Last updated: October 25, 2023

- Written to download monthly agricultural production data from https://nube.siap.gob.mx/avance_agricola/

- TODO: ensure that duplicate excel files are not being downloaded


In [ ]:
import re
import os
from html import unescape
from selenium import webdriver
import time
import pandas as pd
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

base_dir = os.path.join(os.path.expanduser("~"),"Dropbox","Projects","Maize_prediction")
data_dir = os.path.join(base_dir, "Data","SIAP_monthly","Input")

chrome_options = Options()
# chrome_options.add_argument("--headless")
# chrome_options.add_argument("--window-size=1920x1080")
chrome_options.add_argument('--ignore-ssl-errors=yes')
chrome_options.add_argument('--ignore-certificate-errors')


chrome_options.add_experimental_option("prefs", {
  "download.default_directory": data_dir,
  "download.prompt_for_download": False
#   "download.directory_upgrade": True,
#   "safebrowsing.enabled": True
    
})


def click_option(option_text,tag_name='option',text_or_attr='text',get_attr='class'):
#     unclicked = True
#     while unclicked == True:
#         try:
#     WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID,option_text)))
    options = driver.find_elements_by_tag_name(tag_name)
    for link in options:
        if text_or_attr == 'text':
            if link.text == option_text:
                link.click()
#                     unclicked = False
#                     break
        else:
            if link.get_attribute(get_attr) == option_text:
                link.click()
#                     unclicked = False
#                     break
#         except:
#             print("stalled clicking")
#             time.sleep(1)
#             pass
            
def find_table():
    WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID,"Resultados-reporte")))
#     found_table = False
#     while found_table == False:
#         try:
    table = driver.find_element_by_id("Resultados-reporte")
    tablehtml = table.get_attribute('outerHTML')
#             found_table = True
#             break
#         except StaleElementReferenceException:
#             print('Stalled')
#             time.sleep(1)
#             pass
            
    return tablehtml

mes_to_num_dict = {'Enero': 1,
 'Febrero': 2,
 'Marzo': 3,
 'Abril': 4,
 'Mayo': 5,
 'Junio': 6,
 'Julio': 7,
 'Agosto': 8,
 'Septiembre': 9,
 'Octubre': 10,
 'Noviembre': 11,
 'Diciembre': 12}

st_to_num_dict = {'Aguascalientes': 1,
 'Baja California': 2,
 'Baja California Sur': 3,
 'Campeche': 4,
 'Coahuila': 5,
 'Colima': 6,
 'Chiapas': 7,
 'Chihuahua': 8,
 'Ciudad de México': 9,
 'Durango': 10,
 'Guanajuato': 11,
 'Guerrero': 12,
 'Hidalgo': 13,
 'Jalisco': 14,
 'México': 15,
 'Michoacán': 16,
 'Morelos': 17,
 'Nayarit': 18,
 'Nuevo León': 19,
 'Oaxaca': 20,
 'Puebla': 21,
 'Querétaro': 22,
 'Quintana Roo': 23,
 'San Luis Potosí': 24,
 'Sinaloa': 25,
 'Sonora': 26,
 'Tabasco': 27,
 'Tamaulipas': 28,
 'Tlaxcala': 29,
 'Veracruz': 30,
 'Yucatán': 31,
 'Zacatecas': 32}

In [2]:
# !pip uninstall selenium  
# !pip install -U selenium==4.2.0
# !pip install -U webdriver-manager

In [3]:
if os.path.isfile(data_dir+'/Avance de Siembras y Cosechas.xls'):
    os.remove(data_dir+'/Avance de Siembras y Cosechas.xls')

driver = webdriver.Chrome(ChromeDriverManager().install(),options=chrome_options)
# driver = webdriver.Chrome(options=chrome_options)
driver.get('https://nube.siap.gob.mx/avance_agricola/')


/tmp/ipykernel_22455/4067234774.py:4: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome(ChromeDriverManager().install(),options=chrome_options)


OSError: [Errno 8] Exec format error: '/home/j/.wdm/drivers/chromedriver/linux64/135.0.7049.84/chromedriver-linux64/THIRD_PARTY_NOTICES.chromedriver'

In [4]:
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID,"opcionDDRMpio4")))
mun = driver.find_element_by_id("opcionDDRMpio4")
clicked_mun = False
while clicked_mun == False:
    try:
        mun.click()
        clicked_mun = True
        break
    except:
        time.sleep(1)

options = driver.find_elements_by_class_name('form-control')
for opt in options:
    if opt.get_attribute('id') == 'anioagric':
        years = opt.text.split('\n')
    elif opt.get_attribute('id') == 'entidad':
        states = opt.text.split('\n')
        
if 'Nacional' in states:
    states.remove('Nacional')
     
time.sleep(6)
# years = years[1:]

/tmp/ipykernel_19019/3468592870.py:2: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  mun = driver.find_element_by_id("opcionDDRMpio4")
/tmp/ipykernel_19019/3468592870.py:12: DeprecationWarning: find_elements_by_class_name is deprecated. Please use find_elements(by=By.CLASS_NAME, value=name) instead
  options = driver.find_elements_by_class_name('form-control')


In [5]:
def check_df_validity(dataframe,crp,mes,year):
    if len(dataframe.columns) == 8:
        dataframe.columns = ['i','Ent','Mun','Sup. Sem.','Sup. Cos.','Sup. Sin.','Prod','Rendi']
        dataframe.drop('i',1,inplace=True)
        dataframe = dataframe[dataframe['Ent'] != 'Total']
        dataframe['Crop'] = crp
        dataframe['Mes'] = mes_to_num_dict[mes]
        dataframe['Year'] = year
        return False, dataframe
    else:
        return True, pd.DataFrame()
    
def redownload_df(data_dir,better_obsname,crp,mes,year,which_table=99):
    selects = driver.find_elements_by_class_name('select')
    for slt in selects:
        if slt.get_attribute('onclick')=='javascript:descargarExcel();':
            try:
                slt.click()
            except:
                print("Download maybe not clicked")
    time.sleep(3)
    if os.path.isfile(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls')):
        if not os.path.isfile(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls.PART')):
            ### Read excel file
            with open(os.path.join(data_dir,'../Avance de Siembras y Cosechas (copy).xls')) as input_file:
                firstNlines=input_file.readlines()[22:28]

            result = {}
            for line in firstNlines:
                match = re.search('<strong>(.*?):</strong> (.*?)<br', line)
                if match:
                    key = unescape(match.group(1))
                    value = unescape(match.group(2))
                    result[key] = value.strip() if not value.isdigit() else int(value)
            if 'Cultivo' in result.keys():
                if result['Cultivo'] != crp:
                    os.remove(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))
                    return 1, pd.DataFrame()
                else:
                    if which_table == 99:
                        if len(pd.read_html(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))) > 1:
                            return len(pd.read_html(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))), pd.DataFrame()
                        else:
                            df = pd.read_html(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))[0]
                    else:
                        df = pd.read_html(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))[which_table]
                    if len(df.columns) < 8:
                        os.rename(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'),
                                os.path.join(data_dir,'Issue',better_obsname+crp+'.xls'))
                        return 1, pd.DataFrame()
                    else:
                        df.columns = ['i','Ent','Mun','Sup. Sem.','Sup. Cos.','Sup. Sin.','Prod','Rendi']
                        df.drop('i',1,inplace=True)
                        df = df[df['Ent'] != 'Total']
                        df['Crop'] = crp
                        df['Mes'] = mes_to_num_dict[mes]
                        df['Year'] = year
                        os.remove(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))
                        time.sleep(0.5)
                        return 1, df
            else:
                os.remove(os.path.join(data_dir,'Avance de Siembras y Cosechas.xls'))
                return 1, pd.DataFrame()
        else:
            return 1, pd.DataFrame()
    else:
        return 1, pd.DataFrame()

In [7]:
for year in years[:]:
    try:
        click_option(year)
    except:
        pass
    for st in states:
        try:
            click_option(st)
        except:
            pass
        
        options = driver.find_elements_by_class_name('form-control')
        for opt in options:
            if opt.get_attribute('id') == 'mesagric':
                months = opt.text.split('\n')
            elif opt.get_attribute('id') == 'municipio':
                muns = opt.text.split('\n')
            elif opt.get_attribute('id') == 'unidMed':
                unidades = opt.text.split('\n')
            elif opt.get_attribute('id') == 'variedad':
                variedades = opt.text.split('\n')
            elif opt.get_attribute('id') == 'cicloProd':
                ciclos = opt.text.split('\n')
            elif opt.get_attribute('id') == 'modalidad':
                irrig_status = opt.text.split('\n')

        irrig_status =  [irrig_stat for irrig_stat in irrig_status if irrig_stat != 'Riego + Temporal']
        ciclos       =  [cic for cic in ciclos if cic != 'Ciclicos - Perennes' and cic != 'Año Agrícola (OI + PV)']

        # if year == '2023':
        #     months.remove('Octubre')
        #     try:
        #         months.remove('Septiembre')
        #     except:
        #         pass

        for mes in months:
            better_obsname = "Y"+str(year)+"S"+str(st_to_num_dict[st])+'M'+str(mes_to_num_dict[mes])            
            if os.path.isfile(os.path.join(data_dir,better_obsname+'.csv')):
                pass
            else:
                try:
                    click_option(mes)
                except:
                    print('Month maybe not clicked')
                try:
                    click_option('Todos')
                except:
                    print("All muns maybe not clicked")

                full_df = pd.DataFrame()
                print(better_obsname)

                for irg_st in irrig_status:
                    try:
                        click_option(irg_st)
                    except:
                        print('Irrigation status maybe not clicked')
                    for cl in ciclos:
                        try:
                            click_option(cl)
                        except:
                            print('Crop cycle maybe not clicked')
                        time.sleep(0.25)
                        options = driver.find_elements_by_class_name('form-control')
                        for opt in options:
                            if opt.get_attribute('id') == 'cultivo':
                                newcrops = opt.text.split('\n')
                        if 'Resumen cultivos' in newcrops:
                            newcrops.remove('Resumen cultivos')
                        for crp in newcrops:
                            time.sleep(0.25)
                            try:
                                click_option(crp)
                            except:
                                pass
                            try:
                                click_option(crp)
                            except:
                                pass
                            try:
                                click_option(crp)
                            except:
                                pass
                            
                            try:
                                click_option('btn btn-primary pull-right',tag_name='input',text_or_attr='attr')
                            except:
                                pass
                            
                            time.sleep(1)
                            try:
                                tables = driver.find_elements_by_id('Resultados-reporte')
                                if len(tables) == 2:
                                    df            = pd.read_html(tables[1].get_attribute('outerHTML'))[0]
                                    is_invalid, df = check_df_validity(df,crp,mes,year)
                                    if is_invalid: 
                                        _, df = redownload_df(data_dir,better_obsname,crp,mes,year,which_table=1)
                                    justplanteddf = pd.read_html(tables[0].get_attribute('outerHTML'))[0]
                                    is_planted_df_invalid, justplanteddf = check_df_validity(justplanteddf,crp,mes,year)
                                    if is_planted_df_invalid: 
                                        _, justplanteddf = redownload_df(data_dir,better_obsname,crp,mes,year,which_table=0)
                                    if len(justplanteddf.columns) > 4 and len(df.columns) > 4:
                                        justplanteddf.rename(columns={'Sup. Sem.':'Just_planted'}, inplace=True)
                                        justplanteddf = justplanteddf[['Ent', 'Mun','Crop', 'Mes', 'Year','Just_planted']]
                                        df = df.merge(justplanteddf, on = ['Ent', 'Mun','Crop', 'Mes', 'Year'], how='outer')

                                elif len(tables) == 1:
                                    df            = pd.read_html(tables[0].get_attribute('outerHTML'))[0]
                                    is_invalid, df = check_df_validity(df,crp,mes,year)
                                    if is_invalid: 
                                        _, df = redownload_df(data_dir,better_obsname,crp,mes,year, which_table=0)
                                elif len(tables) > 2:
                                    raise Exception
                                else:
                                    print("Table not found")
                                    num_tables, df = redownload_df(data_dir,better_obsname,crp,mes,year)
                                    if num_tables > 1:
                                        _, df = redownload_df(data_dir,better_obsname,crp,mes,year,which_table=1)
                                        _, justplanteddf = redownload_df(data_dir,better_obsname,crp,mes,year,which_table=0)
                                        if len(justplanteddf.columns) > 4 and len(df.columns) > 4:
                                            justplanteddf.rename(columns={'Sup. Sem.':'Just_planted'}, inplace=True)
                                            justplanteddf = justplanteddf[['Ent', 'Mun','Crop', 'Mes', 'Year','Just_planted']]
                                            df = df.merge(justplanteddf, on = ['Ent', 'Mun','Crop', 'Mes', 'Year'], how='outer')
                                        
                            except:
                                print("Table not found")
                                num_tables, df = redownload_df(data_dir,better_obsname,crp,mes,year)
                                if num_tables > 1:
                                    _, df = redownload_df(data_dir,better_obsname,crp,mes,year,which_table=1)
                                    _, justplanteddf = redownload_df(data_dir,better_obsname,crp,mes,year,which_table=0)
                                    if len(justplanteddf.columns) > 4 and len(df.columns) > 4:
                                        justplanteddf.rename(columns={'Sup. Sem.':'Just_planted'}, inplace=True)
                                        justplanteddf = justplanteddf[['Ent', 'Mun','Crop', 'Mes', 'Year','Just_planted']]
                                        df = df.merge(justplanteddf, on = ['Ent', 'Mun','Crop', 'Mes', 'Year'], how='outer')
                            
                            df['Irrig'] = irg_st
                            df['Cycle'] = cl
                            full_df = pd.concat([full_df,df])
                                            
                full_df.to_csv(os.path.join(data_dir,better_obsname+'.csv'),index=False, encoding='utf8')

/tmp/ipykernel_19019/392450937.py:36: DeprecationWarning: find_elements_by_tag_name is deprecated. Please use find_elements(by=By.TAG_NAME, value=name) instead
  options = driver.find_elements_by_tag_name(tag_name)
/tmp/ipykernel_19019/1520723656.py:12: DeprecationWarning: find_elements_by_class_name is deprecated. Please use find_elements(by=By.CLASS_NAME, value=name) instead
  options = driver.find_elements_by_class_name('form-control')


Y2023S1M9
Irrigation status maybe not clicked


/tmp/ipykernel_19019/1520723656.py:65: DeprecationWarning: find_elements_by_class_name is deprecated. Please use find_elements(by=By.CLASS_NAME, value=name) instead
  options = driver.find_elements_by_class_name('form-control')
/tmp/ipykernel_19019/1520723656.py:93: DeprecationWarning: find_elements_by_id is deprecated. Please use find_elements(by=By.ID, value=id_) instead
  tables = driver.find_elements_by_id('Resultados-reporte')
/tmp/ipykernel_19019/780597631.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  dataframe.drop('i',1,inplace=True)
/tmp/ipykernel_19019/780597631.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe['Crop'] = crp
/tmp/ip

Crop cycle maybe not clicked


/tmp/ipykernel_19019/1520723656.py:65: DeprecationWarning: find_elements_by_class_name is deprecated. Please use find_elements(by=By.CLASS_NAME, value=name) instead
  options = driver.find_elements_by_class_name('form-control')
/tmp/ipykernel_19019/1520723656.py:93: DeprecationWarning: find_elements_by_id is deprecated. Please use find_elements(by=By.ID, value=id_) instead
  tables = driver.find_elements_by_id('Resultados-reporte')
/tmp/ipykernel_19019/780597631.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  dataframe.drop('i',1,inplace=True)
/tmp/ipykernel_19019/780597631.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe['Crop'] = crp
/tmp/ip

...

/tmp/ipykernel_19019/780597631.py:37: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop('i',1,inplace=True)
/tmp/ipykernel_19019/1520723656.py:139: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_df = full_df.append(df,sort=False)
/tmp/ipykernel_19019/392450937.py:36: DeprecationWarning: find_elements_by_tag_name is deprecated. Please use find_elements(by=By.TAG_NAME, value=name) instead
  options = driver.find_elements_by_tag_name(tag_name)
/tmp/ipykernel_19019/1520723656.py:93: DeprecationWarning: find_elements_by_id is deprecated. Please use find_elements(by=By.ID, value=id_) instead
  tables = driver.find_elements_by_id('Resultados-reporte')
/tmp/ipykernel_19019/780597631.py:14: DeprecationWarning: find_elements_by_class_name is deprecated. Please use find_elements(by=By.CLASS_NAME, value=name) instea

Table not found


/tmp/ipykernel_19019/780597631.py:37: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop('i',1,inplace=True)
/tmp/ipykernel_19019/1520723656.py:139: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_df = full_df.append(df,sort=False)
/tmp/ipykernel_19019/392450937.py:36: DeprecationWarning: find_elements_by_tag_name is deprecated. Please use find_elements(by=By.TAG_NAME, value=name) instead
  options = driver.find_elements_by_tag_name(tag_name)
/tmp/ipykernel_19019/1520723656.py:93: DeprecationWarning: find_elements_by_id is deprecated. Please use find_elements(by=By.ID, value=id_) instead
  tables = driver.find_elements_by_id('Resultados-reporte')
/tmp/ipykernel_19019/780597631.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword

Table not found


/tmp/ipykernel_19019/780597631.py:37: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop('i',1,inplace=True)
/tmp/ipykernel_19019/1520723656.py:139: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_df = full_df.append(df,sort=False)
/tmp/ipykernel_19019/392450937.py:36: DeprecationWarning: find_elements_by_tag_name is deprecated. Please use find_elements(by=By.TAG_NAME, value=name) instead
  options = driver.find_elements_by_tag_name(tag_name)
